# Modul 2: Trening policy locomotion na GPU (IsaacLab)

Ten notebook pokazuje **pipeline treningu RL** i **analize wynikow**.

**Pelny trening wymaga IsaacLab (Docker Tier 3)** — 4096 rownoleglych srodowisk na GPU
(RTX 3090+ lub chmura). Ten notebook mozna uruchomic w Colab (CPU/T4) do analizy
wytrenowanych modeli i zrozumienia pipeline'u.

**Co znajdziesz w tym notebooku:**
1. Pipeline treningu RL — od konfiguracji po deployment
2. Konfiguracja treningu (observation space, action space, reward weights)
3. Analiza wytrenowanej policy (ONNX inference)
4. Metryki treningu — co monitorowac w TensorBoard
5. Domain Randomization — klucz do sim2real
6. Sim2Sim walidacja

In [ ]:
# Instalacja zaleznosci (analiza, nie trening)
!pip install -q mujoco mujoco_menagerie numpy matplotlib onnxruntime

## Pipeline treningu RL

```
                    PIPELINE: Train → Play → Sim2Sim → Sim2Real

   ┌─────────────────────────────────────────────────────────────────────────┐
   │                           IsaacLab (GPU)                               │
   │                                                                         │
   │  ┌──────────────┐    PPO (rsl_rl)    ┌──────────────┐                  │
   │  │ 4096 robotow │ ──────────────────► │ policy.onnx  │                  │
   │  │ rownolegly   │  ~2000-5000 iter    │ deploy.yaml  │                  │
   │  │ sim na GPU   │  ~1-3h RTX 3090     └──────┬───────┘                  │
   │  └──────────────┘                            │                          │
   │        │                                     │                          │
   │        ▼                                     │                          │
   │  TensorBoard / WandB                         │                          │
   │  (monitorowanie)                             │                          │
   └──────────────────────────────────────────────┼──────────────────────────┘
                                                  │
                          ┌────────────────────────┤
                          │                        │
                          ▼                        ▼
                   ┌─────────────┐          ┌─────────────┐
                   │ Sim2Sim     │          │ Sim2Real    │
                   │ Mujoco +    │  ─────►  │ G1 Robot +  │
                   │ g1_ctrl     │  waliduj  │ Jetson Orin │
                   └─────────────┘          └─────────────┘
                        │
                        ▼
                  trajectory.json
                  (browser simulator)
```

**Kluczowe etapy:**
1. **Train**: PPO z 4096 rownolegymi srodowiskami, ~200K kroków/s na RTX 3090
2. **Play**: Walidacja w IsaacLab z wizualizacja
3. **Sim2Sim**: Transfer do Mujoco (inny symulator = test generalizacji)
4. **Sim2Real**: Deployment na fizycznym robocie G1

## Konfiguracja treningu

Trening RL w unitree_rl_lab wymaga dwoch plikow konfiguracji:

1. **`rsl_rl_ppo_cfg.py`** — parametry algorytmu PPO (learning rate, gamma, clipping, siec)
2. **`velocity_env_cfg.py`** — srodowisko: obserwacje, akcje, nagrody, terrain, domain randomization

Ponizej kluczowe parametry jako slownik Pythona.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─── Konfiguracja treningu (odzwierciedla unitree_rl_lab) ───

training_config = {
    # === Algorytm PPO ===
    'ppo': {
        'learning_rate': 1e-3,
        'gamma': 0.99,              # discount factor
        'lam': 0.95,                # GAE lambda
        'clip_param': 0.2,          # PPO clipping epsilon
        'num_learning_epochs': 5,   # epochs per PPO update
        'num_mini_batches': 4,
        'value_loss_coef': 1.0,
        'entropy_coef': 0.01,       # zacheta do eksploracji
    },

    # === Siec neuronowa ===
    'policy_network': {
        'actor_hidden_dims': [512, 256, 128],
        'critic_hidden_dims': [512, 256, 128],
        'activation': 'elu',
        'init_noise_std': 1.0,      # poczatkowy szum eksploracji
    },

    # === Observation space ===
    'observations': {
        'actor_dim': 45,
        'critic_dim': 48,           # + base_lin_vel (uprzywilejowane)
        'history_length': 5,        # 5 ostatnich obs-action par
        'effective_input_dim': 45 * 5,  # = 225
        'components': {
            'base_ang_vel': {'dim': 3, 'scale': 0.25, 'noise': 0.2},
            'projected_gravity': {'dim': 3, 'scale': 1.0, 'noise': 0.05},
            'velocity_commands': {'dim': 3, 'scale': 1.0},
            'joint_pos_rel': {'dim': 12, 'scale': 1.0, 'noise': 0.01},
            'joint_vel_rel': {'dim': 12, 'scale': 0.05, 'noise': 1.5},
            'last_action': {'dim': 12, 'scale': 1.0},
        },
    },

    # === Action space ===
    'actions': {
        'dim': 12,                  # 12 stawow nog (6 per noga)
        'type': 'delta_joint_positions',
        'action_scale': 0.25,       # target_q = default_q + action * 0.25
        'clip_range': [-1.0, 1.0],
        'controlled_joints': [
            'left_hip_pitch', 'left_hip_roll', 'left_hip_yaw',
            'left_knee', 'left_ankle_pitch', 'left_ankle_roll',
            'right_hip_pitch', 'right_hip_roll', 'right_hip_yaw',
            'right_knee', 'right_ankle_pitch', 'right_ankle_roll',
        ],
    },

    # === Nagrody (wagi) ===
    'rewards': {
        'tracking_lin_vel': {'weight': 1.5, 'type': '+', 'kernel': 'exp', 'sigma': 0.25},
        'tracking_ang_vel': {'weight': 0.75, 'type': '+', 'kernel': 'exp', 'sigma': 0.25},
        'alive': {'weight': 2.0, 'type': '+'},
        'base_height': {'weight': -15.0, 'type': '-', 'target': 0.78},
        'orientation': {'weight': -5.0, 'type': '-'},
        'action_rate': {'weight': -0.01, 'type': '-'},
        'joint_acc': {'weight': -2.5e-7, 'type': '-'},
        'dof_pos_limits': {'weight': -10.0, 'type': '-'},
        'energy': {'weight': -0.001, 'type': '-'},
    },

    # === Srodowisko ===
    'environment': {
        'num_envs': 4096,           # roboty rownolegly
        'episode_length': 1000,     # krokow na epizod
        'dt': 0.002,                # timestep fizyki
        'control_decimation': 10,   # policy co 10 krokow = 50 Hz
        'policy_dt': 0.02,          # efektywny dt policy
    },

    # === Trening ===
    'training': {
        'max_iterations': 5000,
        'save_interval': 100,
        'log_interval': 10,
        'expected_time_rtx3090': '1-3h',
    },
}

# Wyswietl podsumowanie
print("=== Konfiguracja treningu G1 Velocity ===")
print(f"\nObservation: {training_config['observations']['actor_dim']}D actor, "
      f"{training_config['observations']['critic_dim']}D critic")
print(f"Effective input (z historia): {training_config['observations']['effective_input_dim']}D")
print(f"Action: {training_config['actions']['dim']}D (delta joint positions)")
print(f"Action scale: {training_config['actions']['action_scale']}")
print(f"\nSiec: Actor {training_config['policy_network']['actor_hidden_dims']}, "
      f"Critic {training_config['policy_network']['critic_hidden_dims']}")
print(f"Aktywacja: {training_config['policy_network']['activation']}")
print(f"\nSrodowisko: {training_config['environment']['num_envs']} robotow, "
      f"policy @ {1/training_config['environment']['policy_dt']:.0f} Hz")
print(f"\nNagrody:")
for name, cfg in training_config['rewards'].items():
    sign = '+' if cfg['weight'] > 0 else '-'
    print(f"  {sign} {name:25s} weight={cfg['weight']:+.4f}")

## Analiza wytrenowanej policy (ONNX)

Po treningu w IsaacLab, policy jest eksportowana do formatu **ONNX** — universalnego
formatu modeli ML, ktory mozna uruchomic na:
- Jetson Orin (robot)
- Mujoco (sim2sim)
- Dowolnym PC z `onnxruntime`

Ponizej schemat inferencji — uzyj po wytrenowaniu policy.

In [ ]:
# ─── Schemat inferencji policy (wymaga policy.onnx z treningu) ───

# Odkomentuj po wytrenowaniu policy w IsaacLab:
#
# import onnxruntime as ort
#
# # Zaladuj model ONNX
# session = ort.InferenceSession("policy.onnx")
# input_name = session.get_inputs()[0].name
# print(f"Input: {input_name}, shape: {session.get_inputs()[0].shape}")
# print(f"Output shape: {session.get_outputs()[0].shape}")
#
# # Inferencja — pojedynczy krok
# obs = np.zeros((1, 45), dtype=np.float32)  # 45D obserwacja
# action = session.run(None, {input_name: obs})[0]
# print(f"Action: {action}")
#
# # Z historia (history_length=5):
# obs_with_history = np.zeros((1, 225), dtype=np.float32)  # 45 * 5
# action = session.run(None, {input_name: obs_with_history})[0]
#
# # Konwersja na pozycje stawow:
# default_q = np.array([0.0] * 12)  # domyslne pozycje
# target_q = default_q + action[0, :12] * 0.25  # action_scale

# Informacje o przestrzeni obserwacji i akcji
print("=== Schemat inferencji policy ===")
print()
print("Observation space: 45D (ang_vel, gravity, commands, joint_pos, joint_vel, last_action)")
print("  Z historia (h=5): 45 * 5 = 225D")
print()
print("Action space: 12D (delta joint positions, zakres [-1, 1])")
print("  Kontrolowane stawy: 6 per noga (hip_pitch/roll/yaw, knee, ankle_pitch/roll)")
print()
print("Action scaling: target_q = default_q + action * 0.25")
print("  PD control: torque = kp * (target_q - q) + kd * (0 - dq)")
print()
print("Petla sterowania:")
print("  1. Odczytaj sensory (IMU, enkodery) → zbuduj obserwacje (45D)")
print("  2. Dolacz do historii (5 ostatnich par obs-action)")
print("  3. Policy inference: action = policy(obs_history) → 12D")
print("  4. Skaluj: target_q = default_q + action * 0.25")
print("  5. Wyslij target_q do kontrolera PD")
print("  6. Powtorz @ 50 Hz")

## Metryki treningu

Podczas treningu monitoruj w TensorBoard / WandB:

| Metryka | Oczekiwane zachowanie | Sygnaly ostrzegawcze |
|---|---|---|
| `mean_reward` | Rosnie stabilnie | Plateau → konkurujace nagrody |
| `episode_length` | Rosnie (robot chodzi dluzej) | Spada → robot upada |
| `policy_loss` | Maleje stabilnie | Oscylacje → za duzy LR |
| `value_loss` | Maleje | Rosnie → critic nie nadaza |
| `mean_tracking_vel` | Zbliża sie do 1.0 | Stale 0 → zbyt wysoka kara |
| `mean_energy` | Maleje lub stabilna | Rosnie → robot drga |
| `std` (action noise) | Maleje powoli | Spada do 0 → za wczesna konwergencja |

**Typowe problemy:**
- Robot stoi w miejscu → zmniejsz kary, zwieksz tracking_vel
- Robot drga → dodaj action_rate penalty, zmniejsz init_noise_std
- Robot upada po 100 krokach → zwieksz alive reward, dodaj curriculum
- Reward hacking → sprawdz czy robot nie exploituje srodowiska

In [ ]:
# ─── Przykladowe krzywe treningowe (syntetyczne) ───
# W prawdziwym treningu te dane pochodza z TensorBoard/WandB

np.random.seed(42)
iterations = np.arange(0, 5001, 10)

# Symulacja typowych krzywych treningowych
def smooth(x, window=50):
    return np.convolve(x, np.ones(window)/window, mode='same')

# Mean reward: rosnie logarytmicznie z szumem
mean_reward_raw = 8 * (1 - np.exp(-iterations / 1500)) + np.random.randn(len(iterations)) * 0.5
mean_reward = smooth(mean_reward_raw, 20)

# Episode length: rosnie (robot chodzi dluzej)
ep_length_raw = 200 + 800 * (1 - np.exp(-iterations / 2000)) + np.random.randn(len(iterations)) * 50
ep_length = smooth(ep_length_raw, 20)
ep_length = np.clip(ep_length, 50, 1000)

# Tracking velocity: wolno rosnie
tracking_raw = 0.8 * (1 - np.exp(-iterations / 2500)) + np.random.randn(len(iterations)) * 0.08
tracking = smooth(tracking_raw, 20)
tracking = np.clip(tracking, 0, 1)

# Energy: maleje
energy_raw = 50 * np.exp(-iterations / 3000) + 10 + np.random.randn(len(iterations)) * 3
energy = smooth(energy_raw, 20)

# Action std: maleje powoli
action_std = 1.0 * np.exp(-iterations / 4000) + 0.15 + np.random.randn(len(iterations)) * 0.02
action_std = smooth(action_std, 30)

# Policy loss: maleje z oscylacjami
policy_loss = 0.05 * np.exp(-iterations / 2000) + 0.005 + np.abs(np.random.randn(len(iterations))) * 0.003
policy_loss = smooth(policy_loss, 30)

# Wykresy
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0, 0].plot(iterations, mean_reward, color='#2196F3', linewidth=1.5)
axes[0, 0].fill_between(iterations, mean_reward - 0.5, mean_reward + 0.5, alpha=0.2, color='#2196F3')
axes[0, 0].set_title('Mean Reward', fontweight='bold')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=7, color='green', linestyle='--', alpha=0.5, label='dobry level')
axes[0, 0].legend()

axes[0, 1].plot(iterations, ep_length, color='#4CAF50', linewidth=1.5)
axes[0, 1].set_title('Episode Length', fontweight='bold')
axes[0, 1].set_ylabel('Kroki')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=1000, color='green', linestyle='--', alpha=0.5, label='max (1000)')
axes[0, 1].legend()

axes[0, 2].plot(iterations, tracking, color='#FF9800', linewidth=1.5)
axes[0, 2].set_title('Tracking Velocity Reward', fontweight='bold')
axes[0, 2].set_ylabel('Reward [0-1]')
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].set_ylim(0, 1.1)

axes[1, 0].plot(iterations, energy, color='#F44336', linewidth=1.5)
axes[1, 0].set_title('Energy Consumption', fontweight='bold')
axes[1, 0].set_ylabel('Energy')
axes[1, 0].set_xlabel('Iteracja')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(iterations, action_std, color='#9C27B0', linewidth=1.5)
axes[1, 1].set_title('Action Std (eksploracja)', fontweight='bold')
axes[1, 1].set_ylabel('Std')
axes[1, 1].set_xlabel('Iteracja')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=0.15, color='red', linestyle='--', alpha=0.5, label='min (za malo eksploracji)')
axes[1, 1].legend()

axes[1, 2].plot(iterations, policy_loss, color='#607D8B', linewidth=1.5)
axes[1, 2].set_title('Policy Loss', fontweight='bold')
axes[1, 2].set_ylabel('Loss')
axes[1, 2].set_xlabel('Iteracja')
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Przykladowe krzywe treningowe PPO — G1 Velocity\n(dane syntetyczne, odwzorowuja typowy przebieg)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretacja:")
print("- Mean reward rosnie logarytmicznie — typowe dla PPO")
print("- Episode length osiaga max ~1000 krokow po ~3000 iteracji")
print("- Tracking velocity rosnie wolniej — wymaga dobrego chodu")
print("- Energy maleje — robot uczy sie efektywnego ruchu")
print("- Action std maleje powoli — eksploracja → eksploatacja")
print("\nW TensorBoard: tensorboard --logdir logs/rsl_rl/G1_velocity/")

## Domain Randomization

Domain Randomization (DR) to **klucz do sim2real transfer**. Polega na randomizacji
parametrow symulacji podczas treningu, aby policy nauczyla sie byc odporna na
roznice miedzy symulacja a rzeczywistoscia.

**Bez DR:** policy dziala idealnie w symulacji, ale upada na prawdziwym robocie.

**Z DR:** policy jest nieco gorsza w symulacji, ale generalizuje na prawdziwy swiat.

> "Szum na obserwacjach to za malo!" — Typowy blad: dodanie +-0.05 rad szumu na joint_pos
> i nazwanie tego domain randomization. Realne silniki maja opoznienia (0-40ms),
> nieliniowe tarcie, backlash. Podloga ma zmienne tarcie. Robot moze niesc cos.

In [ ]:
# ─── Parametry Domain Randomization ───

dr_params = {
    'Robot': [
        ('Masa linkow', '+-15-30%', 'Zmienia dynamike chodu'),
        ('Offset CoM', '+-0.01-0.05 m', 'Symuluje ladunki, kable'),
        ('Tarcie stawow', '0-0.5 Nm', 'Realne stawy maja tarcie'),
    ],
    'Aktuatory': [
        ('Sila silnikow', '80-120%', 'Rozrzut miedzy egzemplarzami'),
        ('PD gains (kp, kd)', '+-20-30%', 'Niepewnosc kalibracji'),
        ('Opoznienie', '0-40 ms', 'Komunikacja, przetwarzanie'),
        ('Tarcie (Coulomb)', '0-0.3 Nm', 'Nieliniowe tarcie silnikow'),
    ],
    'Srodowisko': [
        ('Tarcie podlogi', '0.3-2.0', 'Kafelki vs dywan vs lod'),
        ('Grawitacja', '+-0.1 m/s^2', 'Kalibracja IMU'),
        ('Zewnetrzne sily', '0-200 N co 3-8s', 'Popchniecja, wiatr'),
    ],
    'Sensory': [
        ('Szum pozycji stawow', '+-0.01-0.05 rad', 'Rozdzielczosc enkoderow'),
        ('Szum predkosci', '+-0.05-0.5 rad/s', 'Numeryczne rozniczkowanie'),
        ('Szum IMU', '+-0.05-0.2 rad/s', 'Drift, kalibracja'),
    ],
}

# Wizualizacja jako tabela
print("=" * 75)
print(f"{'Kategoria':<14} {'Parametr':<25} {'Zakres':<18} {'Dlaczego?'}")
print("=" * 75)
for category, params in dr_params.items():
    for i, (param, range_str, why) in enumerate(params):
        cat_str = category if i == 0 else ''
        print(f"{cat_str:<14} {param:<25} {range_str:<18} {why}")
    print("-" * 75)

# Wizualizacja wplywu DR na tracking velocity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

np.random.seed(42)
iterations = np.arange(0, 3001, 10)

# Bez DR — szybko converguje, ale overfit
no_dr = 0.95 * (1 - np.exp(-iterations / 800)) + np.random.randn(len(iterations)) * 0.03
no_dr = np.clip(smooth(no_dr, 15), 0, 1)

# Z DR — wolniej, ale generalizuje
with_dr = 0.82 * (1 - np.exp(-iterations / 1500)) + np.random.randn(len(iterations)) * 0.05
with_dr = np.clip(smooth(with_dr, 15), 0, 1)

axes[0].plot(iterations, no_dr, label='Bez DR', color='#F44336', linewidth=2)
axes[0].plot(iterations, with_dr, label='Z DR', color='#4CAF50', linewidth=2)
axes[0].set_title('Tracking Velocity w symulacji', fontweight='bold')
axes[0].set_xlabel('Iteracja')
axes[0].set_ylabel('Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].annotate('Bez DR szybciej\nconverguje w sim', xy=(1500, 0.92),
                 fontsize=9, color='#F44336')

# Sim2Real gap
categories = ['Sim (trening)', 'Sim2Sim\n(Mujoco)', 'Sim2Real\n(robot)']
no_dr_transfer = [0.95, 0.60, 0.20]
with_dr_transfer = [0.82, 0.78, 0.70]

x = np.arange(len(categories))
width = 0.35

bars1 = axes[1].bar(x - width/2, no_dr_transfer, width, label='Bez DR', color='#F44336', alpha=0.8)
bars2 = axes[1].bar(x + width/2, with_dr_transfer, width, label='Z DR', color='#4CAF50', alpha=0.8)
axes[1].set_title('Transfer performance', fontweight='bold')
axes[1].set_ylabel('Tracking Velocity')
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, 1.1)

# Adnotacje
for bar, val in zip(bars1, no_dr_transfer):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}',
                 ha='center', fontsize=9, fontweight='bold')
for bar, val in zip(bars2, with_dr_transfer):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}',
                 ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Wplyw Domain Randomization na transfer (dane pogladowe)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Wniosek: DR obniza performance w symulacji, ale dramatycznie poprawia sim2real.")
print("Policy bez DR moze upasc po pierwszym kroku na realnym robocie.")

## Sim2Sim walidacja

Przed wdrozeniem na fizycznym robocie, walidujemy policy w **Mujoco** (innym symulatorze niz
IsaacLab/PhysX). Jesli policy dziala dobrze w obu symulatorach, jest wieksza szansa
ze zadziala na robocie.

Dwa sposoby:

### 1. Pelny sim2sim z g1_ctrl (unitree_mujoco)
```bash
# Terminal 1: uruchom symulator Mujoco
~/unitree_mujoco/simulate/build/unitree_mujoco

# Terminal 2: uruchom kontroler z policy
~/unitree_rl_lab/deploy/robots/g1_23dof/build/g1_ctrl

# Klawisze w Mujoco:
#   x = stand (wstaj)
#   8 = opusc (squat)
#   v = policy (uruchom policy RL)
#   9 = odczep (free fall — test stabilnosci)
```

### 2. Export trajektorii do JSON (browser simulator)
```bash
python exercises/module2/export_trajectory.py \
    --policy policy.onnx \
    --duration 5.0 \
    --vx 0.5 --wz 0.0
```

In [ ]:
# ─── Schemat uzycia export_trajectory.py ───

# Po wytrenowaniu policy w IsaacLab:
#
# 1. Znajdz plik policy.onnx w katalogu logow:
#    ls logs/rsl_rl/G1_velocity/*/exported/
#
# 2. Export trajektorii:
#    python exercises/module2/export_trajectory.py \
#        --policy logs/rsl_rl/G1_velocity/2024-01-01_12-00-00/exported/policy.onnx \
#        --duration 5.0 \
#        --vx 0.5 --vy 0.0 --wz 0.0 \
#        --output trajectory_walk.json
#
# 3. Rozne tryby ruchu:
#    --vx 0.5              # chod do przodu
#    --vx 0.5 --wz 0.3    # chod z skretem
#    --vx 0.0 --wz 0.5    # obrot w miejscu
#    --vx 1.0              # szybki chod
#
# 4. Upload trajectory.json do symulatora przegladarkowego:
#    Tab 'Trajektoria' → Upload → wybierz plik

print("=== export_trajectory.py — schemat uzycia ===")
print()
print("Wymagania:")
print("  - policy.onnx (z treningu IsaacLab)")
print("  - mujoco, onnxruntime, numpy")
print("  - model G1 (mujoco_menagerie lub scene.xml)")
print()
print("Parametry domyslne (matching unitree_rl_lab G1-23dof-Velocity):")

default_config = {
    'num_observations': 45,
    'num_actions': 12,
    'action_scale': 0.25,
    'control_decimation': 10,
    'dt': 0.002,
    'policy_dt': 0.02,
    'history_length': 1,  # auto-detect z policy input size
    'obs_scales': {
        'ang_vel': 0.25,
        'dof_pos': 1.0,
        'dof_vel': 0.05,
    },
}

for key, val in default_config.items():
    print(f"  {key}: {val}")

print()
print("Kontrolowane stawy (12):")
joints = [
    'left_hip_pitch', 'left_hip_roll', 'left_hip_yaw',
    'left_knee', 'left_ankle_pitch', 'left_ankle_roll',
    'right_hip_pitch', 'right_hip_roll', 'right_hip_yaw',
    'right_knee', 'right_ankle_pitch', 'right_ankle_roll',
]
for i, j in enumerate(joints):
    side = 'Lewa noga' if i < 6 else 'Prawa noga'
    if i == 0 or i == 6:
        print(f"  {side}:")
    print(f"    [{i:2d}] {j}")

print()
print("Format wyjsciowy (trajectory.json):")
print('  {"name": "...", "dt": 0.02, "nq": 37, "frames": [[qpos_t0], [qpos_t1], ...]}')

## Nastepne kroki

### Pelny trening (Docker Tier 3)

Aby wytrenowac policy lokomotion potrzebujesz:
1. **GPU**: RTX 3090+ (minimum), RTX 4090 (rekomendowane)
2. **IsaacLab**: Docker container z NVIDIA Isaac Sim
3. **unitree_rl_lab**: Framework treningowy Unitree

```bash
# Uruchom Docker z IsaacLab
docker run --gpus all -it isaaclab:latest bash

# Trening
cd ~/unitree_rl_lab
./unitree_rl_lab.sh -t --task Unitree-G1-23dof-Velocity

# Monitorowanie
tensorboard --logdir logs/rsl_rl/G1_velocity/
```

### Modul 3: Manipulacja

Po opanowaniu lokomotion, Modul 3 uczy sterowania **ramionami** G1:
- Inverse kinematics (IK) z Pinocchio
- Manipulacja obiektami
- Koordynacja chod + chwytanie

### Zasoby
- [unitree_rl_lab](https://github.com/unitreerobotics/unitree_rl_lab) — framework treningowy
- [IsaacLab](https://isaac-sim.github.io/IsaacLab/) — srodowisko symulacyjne NVIDIA
- [rsl_rl](https://github.com/leggedrobotics/rsl_rl) — implementacja PPO
- [mujoco_menagerie](https://github.com/google-deepmind/mujoco_menagerie) — modele robotow